# Market Basket — Basic Statistical Analysis

Dataset: [frtgnn/dunnhumby-the-complete-journey](https://www.kaggle.com/datasets/frtgnn/dunnhumby-the-complete-journey) (Dunnhumby "The Complete Journey").

This notebook runs a **basic statistical analysis** of grocery-shopping transactions
(`transaction_data.csv`, ~2.6M rows):

- data shape, types, missing values
- transaction-level descriptive statistics (quantity, sales value, discounts)
- basket-level statistics (items per basket, basket value)
- top products, top stores, top households
- day-of-week / week / hour patterns
- discount usage
- correlations between continuous variables

Everything is descriptive; no modeling. Artifacts are written to `/kaggle/working`
and pulled back with `kaggle kernels output`.

In [ ]:
import os
import glob
import json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

print("pandas", pd.__version__, "| numpy", np.__version__)

WORK = "/kaggle/working"
os.makedirs(WORK, exist_ok=True)

# Locate the mounted dataset (layout has changed before; glob to stay robust).
hits = sorted(glob.glob("/kaggle/input/**/transaction_data.csv", recursive=True))
print("transaction_data.csv at:", hits)
if not hits:
    raise SystemExit("transaction_data.csv not mounted")
CSV = hits[0]

In [ ]:
df = pd.read_csv(CSV)
print("shape:", df.shape)
print("columns:", list(df.columns))
print(df.dtypes.to_string())
df.head().to_string()

In [ ]:
print("=== missing values ===")
missing = df.isna().sum()
print(missing[missing > 0].to_string() if missing.any() else "none")

n_neg_sales = int((df["SALES_VALUE"] < 0).sum())
n_neg_qty = int((df["QUANTITY"] < 0).sum())
print(f"\nnegative SALES_VALUE rows (returns): {n_neg_sales:,} "
      f"({n_neg_sales / len(df):.2%})")
print(f"negative QUANTITY rows (returns):    {n_neg_qty:,}")

n_households = df["household_key"].nunique()
n_baskets = df["BASKET_ID"].nunique()
n_products = df["PRODUCT_ID"].nunique()
n_stores = df["STORE_ID"].nunique()
print(f"\nhouseholds: {n_households:,} | baskets: {n_baskets:,} | "
      f"products: {n_products:,} | stores: {n_stores:,}")

In [ ]:
num_cols = ["QUANTITY", "SALES_VALUE", "RETAIL_DISC", "TRANS_TIME",
            "WEEK_NO", "COUPON_DISC", "COUPON_MATCH_DISC", "DAY"]
desc = df[num_cols].describe().T
desc["missing"] = df[num_cols].isna().sum()
desc.to_csv(f"{WORK}/descriptive_stats.csv")
desc.to_string()

In [ ]:
# Basket-level statistics
basket = (df.groupby("BASKET_ID", sort=False)
            .agg(n_items=("PRODUCT_ID", "nunique"),
                 n_units=("QUANTITY", "sum"),
                 value=("SALES_VALUE", "sum"))
            .reset_index())

print("=== basket-level ===")
print(f"baskets: {len(basket):,}")
print(basket[["n_items", "n_units", "value"]].describe().to_string())

fig, ax = plt.subplots(figsize=(8, 4))
basket[basket["n_items"] <= 80]["n_items"].hist(bins=40, ax=ax, color="#2a6dd4")
ax.set_title("Items per basket (clipped at 80)")
ax.set_xlabel("distinct items per basket")
fig.tight_layout()
fig.savefig(f"{WORK}/items_per_basket.png", dpi=110)
plt.close(fig)
print("saved items_per_basket.png")

In [ ]:
# Sales value distribution per transaction
sales = df["SALES_VALUE"]
print("=== SALES_VALUE (per transaction row) ===")
print(sales.describe().to_string())
print(f"zero-sales rows: {(sales == 0).sum():,} ({(sales == 0).mean():.2%})")

fig, ax = plt.subplots(figsize=(8, 4))
sales[(sales > 0) & (sales <= 100)].hist(bins=50, ax=ax, color="#1c7e4a")
ax.set_title("SALES_VALUE per line item (0 < value <= 100)")
ax.set_xlabel("SALES_VALUE ($)")
fig.tight_layout()
fig.savefig(f"{WORK}/sales_distribution.png", dpi=110)
plt.close(fig)
print("saved sales_distribution.png")

In [ ]:
# Top products
prod = (df.groupby("PRODUCT_ID", sort=False)
          .agg(n_transactions=("BASKET_ID", "nunique"),
               n_units=("QUANTITY", "sum"),
               sales=("SALES_VALUE", "sum"))
          .sort_values("sales", ascending=False)
          .head(20)
          .reset_index())
prod["PRODUCT_ID"] = prod["PRODUCT_ID"].astype(str)
prod.to_csv(f"{WORK}/top_products.csv", index=False)
print(prod.to_string(index=False))

In [ ]:
# Top stores
stores = (df.groupby("STORE_ID", sort=False)
            .agg(n_transactions=("BASKET_ID", "nunique"),
                 n_units=("QUANTITY", "sum"),
                 sales=("SALES_VALUE", "sum"))
            .sort_values("sales", ascending=False)
            .head(20)
            .reset_index())
stores.to_csv(f"{WORK}/top_stores.csv", index=False)
print(stores.to_string(index=False))

In [ ]:
# Top households
hh = (df.groupby("household_key", sort=False)
        .agg(n_baskets=("BASKET_ID", "nunique"),
             n_units=("QUANTITY", "sum"),
             sales=("SALES_VALUE", "sum"))
        .reset_index())
hh["avg_basket_value"] = hh["sales"] / hh["n_baskets"]
hh_top = hh.sort_values("sales", ascending=False).head(20)
hh_top.to_csv(f"{WORK}/top_households.csv", index=False)
print(hh_top.to_string(index=False))

In [ ]:
# NOTE: in this dataset DAY is a running day-of-study counter (1..711) over ~two
# calendar years, NOT a day-of-week. A true weekday breakdown would need a calendar
# mapping that the dataset does not provide, so we only describe the running-day
# counter and the calendar week (WEEK_NO 1..102) volume.
by_day = (df.groupby("DAY")
            .agg(rows=("BASKET_ID", "size"),
                 baskets=("BASKET_ID", "nunique"),
                 sales=("SALES_VALUE", "sum"))
            .reset_index())
by_day.to_csv(f"{WORK}/sales_by_day.csv", index=False)
print(f"DAY is a running study-day counter: {int(df['DAY'].min())}..{int(df['DAY'].max())}")
print(f"rows per running day: median {int(by_day['rows'].median())}, max {int(by_day['rows'].max())}")
print(by_day.head().to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(by_day["DAY"], by_day["sales"], color="#b3542a")
ax.set_title("Sales by running DAY counter (study day)")
ax.set_xlabel("DAY")
ax.set_ylabel("SALES_VALUE ($)")
fig.tight_layout()
fig.savefig(f"{WORK}/sales_by_day.png", dpi=110)
plt.close(fig)

# Week-of-year volume (weekday to calendar week is explicit in WEEK_NO)
by_week = df.groupby("WEEK_NO", sort=False).size()
fig, ax = plt.subplots(figsize=(8, 3))
by_week.plot(ax=ax, color="#5a3fb5")
ax.set_title("Transactions per WEEK_NO (calendar week 1..102)")
ax.set_xlabel("WEEK_NO")
fig.tight_layout()
fig.savefig(f"{WORK}/transactions_per_week.png", dpi=110)
plt.close(fig)
print("saved sales_by_day.png, transactions_per_week.png")

In [ ]:
# Hour-of-day (TRANS_TIME is HHMM integer)
df["HOUR"] = df["TRANS_TIME"] // 100
by_hour = df.groupby("HOUR", sort=False).agg(rows=("BASKET_ID", "size"),
                                             sales=("SALES_VALUE", "sum")).reset_index()
by_hour.to_csv(f"{WORK}/sales_by_hour.csv", index=False)
print(by_hour.to_string(index=False))

In [ ]:
# Discount usage
n_ret = int((df["RETAIL_DISC"] < 0).sum())
n_coupon = int((df["COUPON_DISC"] > 0).sum())
n_cm = int((df["COUPON_MATCH_DISC"] > 0).sum())
print(f"line items with retail discount : {n_ret:,} ({n_ret / len(df):.2%})")
print(f"line items with coupon discount : {n_coupon:,} ({n_coupon / len(df):.2%})")
print(f"line items with coupon match    : {n_cm:,} ({n_cm / len(df):.2%})")

# Correlation among continuous variables
corr = df[num_cols].corr()
corr.to_csv(f"{WORK}/correlations.csv")
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="RdYlBu", vmin=-1, vmax=1)
ax.set_xticks(range(corr.shape[0])); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(corr.shape[0])); ax.set_yticklabels(corr.columns)
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Correlation of transaction variables")
fig.tight_layout()
fig.savefig(f"{WORK}/correlation_heatmap.png", dpi=110)
plt.close(fig)

print(corr.round(2).to_string())

## Results

The headline figures are written to `/kaggle/working/summary.json` and a plain-text
`ANALYSIS_SUMMARY.md`; all CSVs and PNGs above are also in `/kaggle/working` and can be
pulled with `kaggle kernels output`.

In [ ]:
WORK = '/kaggle/working'
summary = dict(rows=int(df.shape[0]),
              transactions=None,
              households=int(n_households),
              baskets=int(n_baskets),
              distinct_products=int(n_products),
              stores=int(n_stores),
              missing_values=dict(df.isna().sum().to_dict()),
              return_rows=int(n_neg_sales),
              avg_items_per_basket=float(basket['n_items'].mean()),
              median_items_per_basket=float(basket['n_items'].median()),
              avg_basket_value=float(basket['value'].mean()),
              median_basket_value=float(basket['value'].median()),
              avg_sales_per_line=float(df['SALES_VALUE'].mean()),
              top_volume_week=int(df.groupby('WEEK_NO').size().idxmax()),
              top_product_ids=[str(x) for x in prod['PRODUCT_ID'].head(3)],
              top_store_ids=[int(x) for x in stores['STORE_ID'].head(3)],
              n_retail_discount_rows=int(n_ret),
              n_coupon_discount_rows=int(n_coupon))
with open(f'{WORK}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

report = ["# Market Basket — Basic Statistical Analysis", ""]
report += [f'- rows: {df.shape[0]:,}', f'- baskets: {n_baskets:,}',
           f'- households: {n_households:,}', f'- distinct products: {n_products:,}',
           f'- stores: {n_stores:,}',
           f'- return (negative-sales) rows: {n_neg_sales:,} ({n_neg_sales / len(df):.2%})',
           f'- avg items per basket: {basket['n_items'].mean():.2f}',
           f'- median basket value: ${basket['value'].median():.2f}',
           f'- avg basket value: ${basket['value'].mean():.2f}',
           f'- avg sales per line item: ${df['SALES_VALUE'].mean():.2f}',
           f'- top-volume calendar week (WEEK_NO): {int(df.groupby('WEEK_NO').size().idxmax())}',
           f'- DAY is a running study-day counter 1..{int(df['DAY'].max())} (not day-of-week)']
with open(f'{WORK}/ANALYSIS_SUMMARY.md', 'w') as f:
    f.write('\n'.join(report))

print('wrote', f'{WORK}/summary.json', f'{WORK}/ANALYSIS_SUMMARY.md')
print('\n===== HEADLINES =====')
for line in report[2:]:
    print(line)